In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plot style
sns.set_style("whitegrid")

# File configuration
file_path = r"Y:\Spinal Stim_Stroke R01\REDCap\SpinalStimStrokeAim2-FunctionalOutcomes_DATA_LABELS_2025-12-11.csv"
subjects_to_remove = ['SS01', 'SS21']

In [ ]:
def load_and_filter_data(path, excluded_subjects):
    df = pd.read_csv(path)
    
    # Filter out specific subjects from Record ID
    if excluded_subjects:
        df = df[~df['Record ID'].isin(excluded_subjects)].copy()
        
    return df

df = load_and_filter_data(file_path, subjects_to_remove)

In [ ]:
# Rename columns
df = df.rename(columns={
    'Which assessment visit is this?': 'Assessment',
    'Average Time (TUG)': 'Average TUG Times',
    'Average Time (Dual Task)': 'Average Dual Task Times'
})

# Map Assessment 
assessment_map = {
    'Baseline': 'BL',
    'Midpoint #1': 'MID18',
    'Midpoint #2': 'MID24',
    'Post #1': 'POST18',
    'Post #2': 'POST24',
    '1 Month F/U': 'MO1FU'
}
df['Assessment'] = df['Assessment'].replace(assessment_map)

# Consolidate 6MWT into a list
mw6_cols = [
    'Minute 1 Distance', 'Minute 2 Distance', 'Minute 3 Distance', 
    'Minute 4 Distance', 'Minute 5 Distance', 'Minute 6 Distance'
]
df['6MWT'] = df[mw6_cols].values.tolist()

# Split dataframe into SSV and FV to create the 'Speed' column structure
# Process SSV Data
df_ssv = df.copy()
df_ssv['Speed'] = 'SSV'
ssv_cols = ['SSV Trial 1 Time', 'SSV Trial 2 Time', 'SSV Trial 3 Time']
df_ssv['10MWT Times'] = df_ssv[ssv_cols].values.tolist()

# Process FV Data
df_fv = df.copy()
df_fv['Speed'] = 'FV'
fv_cols = ['FV Trial 1 Time', 'FV Trial 2 Time', 'FV Trial 3 Time']
df_fv['10MWT Times'] = df_fv[fv_cols].values.tolist()

# Combine SSV and FV dataframes
final_df = pd.concat([df_ssv, df_fv], ignore_index=True)

# Define final column order
final_cols = [
    'Record ID',
    'Assessment',
    'Speed',
    '10MWT Times',
    '6MWT',
    'Average TUG Times',
    'Average Dual Task Times',
    'P Words',
    'S Words'
]

# Final Dataframe
clean_df = final_df[final_cols]

In [ ]:
# Check the head of the data
pd.set_option('display.max_colwidth', None)
clean_df.head()

In [ ]:
# Order Assessments
assessment_order = ['BL', 'MID18', 'MID24', 'POST18', 'POST24', 'MO1FU']
clean_df['Assessment'] = pd.Categorical(clean_df['Assessment'], categories=assessment_order, ordered=True)
clean_df = clean_df.sort_values(['Record ID', 'Assessment'])

# Process 10MWT (Calculate Mean and SD from the list of trials)
def get_list_stats(x):
    if isinstance(x, list) and len(x) > 0:
        # Filter out 0s or NaNs 
        clean_x = [float(i) for i in x if i is not None]
        if not clean_x: return pd.Series([np.nan, np.nan])
        return pd.Series([np.mean(clean_x), np.std(clean_x)])
    return pd.Series([np.nan, np.nan])

clean_df[['10MWT_Mean', '10MWT_SD']] = clean_df['10MWT Times'].apply(get_list_stats)

# Process 6MWT (Sum the minute distances for Total Distance) 
def get_last(x):
    if isinstance(x, list) and len(x) > 0:
        return float(x[-1])
    return np.nan

clean_df['6MWT_Total'] = clean_df['6MWT'].apply(get_last)


# Dual Task Normalization
# Normalization: Time per Word (Seconds / Word)
clean_df['DT_Norm_P'] = clean_df['Average Dual Task Times'] / clean_df['P Words'].replace(0, np.nan)
clean_df['DT_Norm_S'] = clean_df['Average Dual Task Times'] / clean_df['S Words'].replace(0, np.nan)

# Dual TUG - TUG Difference
clean_df['DT_TUG_Cost'] = clean_df['Average Dual Task Times'] - clean_df['Average TUG Times']


print("Data Engineering Complete.")
clean_df.head()

In [ ]:
subjects = clean_df['Record ID'].unique()

def plot_trajectory_with_errors(data, x_col, y_col, y_err_col=None, title="", ylabel=""):
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Plot each subject
    for subject in data['Record ID'].unique():
        subj_data = data[data['Record ID'] == subject].dropna(subset=[y_col])
        
        if subj_data.empty:
            continue
            
        x = subj_data[x_col]
        y = subj_data[y_col]
        
        # Plot Line
        ax.plot(x, y, marker='o', label=subject, alpha=0.7)
        
        # Add Error Bars if requested (e.g., for 10MWT trial variability)
        if y_err_col:
            yerr = subj_data[y_err_col]
            ax.errorbar(x, y, yerr=yerr, fmt='none', capsize=5, color='gray', alpha=0.5)

    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.set_xlabel('Assessment')
    
    # Place legend outside to avoid clutter
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Subject')
    plt.tight_layout()
    plt.show()

# 10MWT (SSV and FV)
for speed_type in ['SSV', 'FV']:
    subset = clean_df[clean_df['Speed'] == speed_type]
    plot_trajectory_with_errors(
        subset, 
        x_col='Assessment', 
        y_col='10MWT_Mean', 
        y_err_col='10MWT_SD', 
        title=f'10MWT - {speed_type} (Mean of Trials ± SD)', 
        ylabel='Time (s)'
    )
unique_visits_df = clean_df[clean_df['Speed'] == 'SSV'].copy()

# 6MWT
plot_trajectory_with_errors(
    unique_visits_df,
    x_col='Assessment',
    y_col='6MWT_Total',
    title='6MWT Total Distance',
    ylabel='Distance (m)'
)

# Average TUG
plot_trajectory_with_errors(
    unique_visits_df,
    x_col='Assessment',
    y_col='Average TUG Times',
    title='Average TUG Time',
    ylabel='Time (s)'
)

# Dual Task Normalized by P Words 
plot_trajectory_with_errors(
    unique_visits_df,
    x_col='Assessment',
    y_col='DT_Norm_P',
    title='Dual Task Normalized by P Words (Time / Word)',
    ylabel='Seconds per Word'
)

# Dual Task Normalized by S Words 
plot_trajectory_with_errors(
    unique_visits_df,
    x_col='Assessment',
    y_col='DT_Norm_S',
    title='Dual Task Normalized by S Words (Time / Word)',
    ylabel='Seconds per Word'
)

# Dual Task – Raw Average TUG Time
plot_trajectory_with_errors(
    unique_visits_df,
    x_col='Assessment',
    y_col='Average Dual Task Times',
    title='Dual Task TUG Time',
    ylabel='Time (s)'
)

# Dual Task – P Words Count
plot_trajectory_with_errors(
    unique_visits_df,
    x_col='Assessment',
    y_col='P Words',
    title='Dual Task – P Words Count',
    ylabel='Word Count'
)

# Dual Task – S Words Count
plot_trajectory_with_errors(
    unique_visits_df,
    x_col='Assessment',
    y_col='S Words',
    title='Dual Task – S Words Count',
    ylabel='Word Count'
)

# Dual Task Cost (Dual TUG - Single TUG)
plot_trajectory_with_errors(
    unique_visits_df,
    x_col='Assessment',
    y_col='DT_TUG_Cost',
    title='Dual Task Difference (Dual TUG − TUG)',
    ylabel='Time Difference (s)'
)
